# Unit 5 exercises: steer like a driver looks ahead

A controller that aims at a point ahead pays for it on a perfect car and earns it back when the commands arrive late.
These exercises build that controller from its geometry, measure the delay that makes it worth having, and test the claims of the lesson on new settings: Stanley against Unit 4 on noise, the lookahead at a delay the lesson skipped, and a better target rule.
Every number here is a number about racekit's **kinematic** car: a model, not the truth.

Run the setup cell first.
Each exercise names the variables your answer must define; fill in every `None` and `"TODO"`.
Functions you write in one exercise are reused in later ones, so run the exercises in order.
The races in Exercises 6–8 take a few seconds each.

In [ ]:
import math
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import KinematicCar
from racekit.measure import noise_aware_better, repeat_races
from racekit.runner import run_race
from racekit.tracks import load_track

AGENTS = Path("../../projects/racekit/agents")
UNIT_04, UNIT_05 = AGENTS / "unit-04.py", AGENTS / "unit-05.py"
workdir = Path(mkdtemp(prefix="roar-unit-05-ex-"))

# racekit's kinematic car (a model, not the truth)
WHEELBASE = 2.875  # m, L
DELTA_MAX = math.radians(70.0)  # the wheel angle at |steer| = 1; delta = -steer * DELTA_MAX


def rms_offset(result):
    # RMS cross-track error over the driving rows (held == 0), as in Unit 4.
    telemetry = result.telemetry
    offset = telemetry["offset"][telemetry["held"] == 0]
    return float(np.sqrt(np.mean(offset ** 2)))

## Exercise 1 - Pure pursuit by hand

Write three functions:

- `to_car_frame(px, py, cx, cy, yaw)`: Unit 2's frame change, the world point (px, py) seen from a car at (cx, cy) with heading yaw, returned as (forward, left);
- `pursuit_curvature(alpha, distance)`: the curvature of the arc that leaves the car along its heading and passes through a point at bearing α (left positive) and straight-line distance d;
- `pursuit_wheel_angle(alpha, distance)`: the wheel angle δ = arctan(L·κ) that drives that arc (use `math.atan2`: `atan` is not on the book's list of callables).

1. A car is at (12.0, −3.0) with heading 150°, and the target is at (2.0, 4.0): work out the pure-pursuit steer for it.
2. Check your curvature on circles: for R = 15, 30 and 60 m, put a car at the origin heading along +x, tangent to the circle of radius R whose centre is (0, R) (a left turn), and a target on that circle at chord distance 10 m. Compute κ·R from your function.

Define:

- `to_car_frame`, `pursuit_curvature`, `pursuit_wheel_angle`: the functions;
- `forward`, `left`: the target in the car frame (m);
- `alpha`, `distance`: its bearing (rad, left positive) and straight-line distance (m);
- `kappa`, `delta`: the arc's curvature (1/m, left positive) and the wheel angle (rad);
- `steer`: the steer command, −δ / δ_max, clipped to [−1, 1];
- `turns_left`: `True` when the car turns left;
- `circle_products`: a list of the three values κ·R, in the order R = 15, 30, 60.

In [ ]:
def to_car_frame(px, py, cx, cy, yaw):
    return None  # TODO: (forward, left)


def pursuit_curvature(alpha, distance):
    return None  # TODO


def pursuit_wheel_angle(alpha, distance):
    return None  # TODO


forward, left = None, None  # TODO
alpha = None  # TODO
distance = None  # TODO
kappa = None  # TODO
delta = None  # TODO
steer = None  # TODO
turns_left = None  # TODO
circle_products = []  # TODO

## Exercise 2 - Where the car looks

The submission's lookahead is L_d = clip(k·v, L_min, L_max) with k = 0.3 s, L_min = 3 m and L_max = 40 m.
Its target is the **first waypoint after the car's segment whose straight-line distance from the car is at least L_d** (waypoint indices wrap round the lap).

Write:

- `lookahead_distance(speed, k=0.3, minimum=3.0, maximum=40.0)`;
- `first_target(xs, ys, segment, px, py, lookahead)`: the index of that waypoint, searching from `segment + 1` onwards, cyclically.

Use `test-circuit` (`load_track("test-circuit")`, with waypoint coordinates `track.x` and `track.y`) and put the car at its start pose, `px, py, yaw = track.start_pose()`, on segment 0.

1. For the speeds 5, 10, 20, 30, 40 and 150 m/s, compute L_d and the target index.
2. Measure the waypoint spacing: the median distance between consecutive waypoints of `test-circuit` (including the one from the last waypoint back to the first).

Define:

- `lookahead_distance`, `first_target`: the functions;
- `lookaheads`: a dict from speed to L_d (m);
- `target_index`: a dict from speed to the target's waypoint index;
- `limited_speeds`: a sorted list of the speeds at which L_min or L_max is the active limit;
- `spacing`: the median waypoint spacing (m);
- `why_distance`: one sentence on why the target is chosen by distance and not by a fixed number of waypoints ahead (think about what a different track generator, or CARLA's waypoints, would do to the second rule).

In [ ]:
def lookahead_distance(speed, k=0.3, minimum=3.0, maximum=40.0):
    return None  # TODO


def first_target(xs, ys, segment, px, py, lookahead):
    return None  # TODO


lookaheads = {}  # TODO
target_index = {}  # TODO
limited_speeds = []  # TODO
spacing = None  # TODO
why_distance = "TODO"

## Exercise 3 - A delay, measured

`KinematicCar(lag_ticks=N)` applies each action N control ticks after it is commanded.

For each N in 0, 1, 2, 3 and 4: make a `KinematicCar(lag_ticks=N)` (no jitter), `reset` it to (0, 0, 0) at v = 30 m/s, and for 20 ticks call `car.advance(throttle, 0.05, 0.0, 0.0, 0.0, 0.05, 10)` (a small steer to the **right**, held from the first tick), with the throttle that balances the drag at 30 m/s, C_DRAG·v² / A_THROTTLE with C_DRAG = 4·10⁻⁴ and A_THROTTLE = 6 (so the car keeps its speed once the throttle arrives; until then it coasts).
Record the car's (x, y, yaw) after every tick.

Define:

- `first_response`: a dict from N to the first tick (1-based) after which the yaw is not zero;
- `blind_distance`: a dict from N to the distance the car has travelled (straight-line, from the origin) by the end of the last tick at which its yaw is still zero (0.0 for N = 0);
- `lateral_after_1s`: a dict from N to the car's y after the 20 ticks (m; negative is right);
- `lag_cost`: a dict from N to `lateral_after_1s[N] - lateral_after_1s[0]` (m);
- `why_it_matters`: one or two sentences: a controller that measures the error now and corrects it at full strength, what happens to it when every correction arrives N ticks late?

In [ ]:
first_response = {}  # TODO
blind_distance = {}  # TODO
lateral_after_1s = {}  # TODO
lag_cost = {}  # TODO
why_it_matters = "TODO"

## Exercise 4 - Pure pursuit is a PD in disguise

Take a straight line, the x axis, and a car at (0, e) with heading ψ (so e is its cross-track error, left positive).
Let the target be the point on the line at x = L_d.

1. With your Exercise 1 functions, compute the exact pure-pursuit wheel angle for ψ = 0, L_d = 9 m and e ∈ {0.02, 0.1, 0.5, 2.0} m.
2. Linearize: for small e and ψ, sin α ≈ −(e + L_d·ψ) / L_d, so δ ≈ −(2L / L_d²)·e − (2L / L_d)·ψ.
   Compute the linear δ for the same cases, and the relative error |exact − linear| / |exact|.
3. Read off the equivalent PD gains: K_E = 2L / L_d² on the cross-track error and K_HEAD = 2L / L_d on the heading error (Unit 4's law is δ = −K_E·e + K_HEAD·ψ_e with ψ_e = −ψ here).
   With the submission's lookahead (k = 0.3 s, clipped to [3, 40] m), tabulate them at 10, 20, 30 and 40 m/s.
4. Find the speed at which pure pursuit's K_E equals Unit 4's K_E = 0.2 rad/m (analytically, then as a number).

Define:

- `equivalent_k_e(lookahead)`, `equivalent_k_head(lookahead)`: the functions;
- `exact_deltas`, `linear_deltas`, `relative_errors`: NumPy arrays over the four values of e, in order;
- `largest_linear_e`: the largest of the four e whose relative error is below 1%;
- `gains_by_speed`: a dict from speed to the pair (K_E, K_HEAD);
- `matching_speed`: the speed (m/s) at which pure pursuit's K_E is 0.2;
- `detuning`: one sentence: what happens to pure pursuit's gains when the car goes faster, and why that helps under a delay.

In [ ]:
def equivalent_k_e(lookahead):
    return None  # TODO


def equivalent_k_head(lookahead):
    return None  # TODO


E_VALUES = np.array([0.02, 0.1, 0.5, 2.0])
exact_deltas = None  # TODO
linear_deltas = None  # TODO
relative_errors = None  # TODO
largest_linear_e = None  # TODO
gains_by_speed = {}  # TODO
matching_speed = None  # TODO
detuning = "TODO"

## Exercise 5 - Stanley at the front axle

Write `stanley_wheel_angle(px, py, yaw, speed, xs, ys, segment, k=3.0)`, the submission's Stanley law:

- the front axle is at (px + L cos yaw, py + L sin yaw);
- project it onto the segments `segment - 4` … `segment + 11` (cyclic indices; the segment from waypoint i to i + 1) exactly as Unit 2 pinned the cross-track error: the foot parameter clamped to [0, 1] chooses the segment (the smallest distance to the clamped foot wins, ties to the first), and the offset is the signed distance to that segment's line, left positive;
- with e_f that offset and ψ_e the wrapped difference between the chosen segment's heading and yaw, δ = ψ_e + atan2(−k·e_f, max(v, 1)).

Use the `oval`.
Put the car on waypoint 100, heading along segment 100 (the direction from waypoint 100 to waypoint 101), with `segment = 100`.

1. Compute δ there at 20 m/s.
2. Move the car 0.5 m to its **left** (perpendicular to its heading) and compute δ at 5, 10, 20 and 30 m/s.
3. Compare with the same law applied at the **rear** axle (project (px, py) instead of the front axle), for the shifted car at 20 m/s.

Define:

- `stanley_wheel_angle`: the function;
- `delta_on_line`: δ for the car on the line at 20 m/s (rad);
- `delta_left_by_speed`: a dict from speed to δ for the shifted car (rad);
- `rear_axle_delta`: the rear-axle variant's δ for the shifted car at 20 m/s (rad);
- `gain_at_30`: Stanley's effective cross-track gain k / v at 30 m/s (rad/m, for small errors);
- `why_softer`: one sentence on why δ for the same error shrinks with speed, and what that has in common with Exercise 4.

In [ ]:
def stanley_wheel_angle(px, py, yaw, speed, xs, ys, segment, k=3.0):
    return None  # TODO


delta_on_line = None  # TODO
delta_left_by_speed = {}  # TODO
rear_axle_delta = None  # TODO
gain_at_30 = None  # TODO
why_softer = "TODO"

## Exercise 6 - How much delay can pure pursuit take?

Race the submission (pure pursuit, `PP_K = 0.3`) with `TARGET_SPEED = 30.0` (make the variant with `write_variant` into `workdir`) on `"test-circuit"`, with `KinematicCar(jitter=0.05, lag_ticks=N)`, `seed=0` and `max_seconds=400.0`, for N = 0, 1, 2 and 3.

Define:

- `pursuit_by_lag`: a dict from N to a tuple (clean, race time in s, RMS cross-track error in m), where clean means finished with 0 collisions;
- `fig`, `ax`: one plot of the RMS against N, with axis labels and a title;
- `rms_growth`: the RMS at N = 3 divided by the RMS at N = 0;
- `weaving_lag`: the smallest N whose race time is more than 1.0 s slower than at N = 0 (`None` if there is none);
- `clean_but`: one sentence on what the race time and the RMS say that "clean" alone does not.

In [ ]:
pursuit_by_lag = {}  # TODO
fig, ax = None, None  # TODO
rms_growth = None  # TODO
weaving_lag = None  # TODO
clean_but = "TODO"

## Exercise 7 - Is Stanley better than Unit 4?

The lesson's probe record shows Stanley with a slightly lower RMS than Unit 4's controller on every track at 20 m/s.
Test that on the `oval` with Unit 1's measurement rule.

Race `agents/unit-04.py` and a variant of `agents/unit-05.py` with `STEERING = "stanley"` (made with `write_variant`), each with `repeat_races` on `"oval"`, seeds 0–4, `car_factory` giving `KinematicCar(jitter=0.05)` (no delay).

Define:

- `unit_04_rms`, `stanley_rms`: lists of the five per-seed RMS cross-track errors (m), seed order;
- `rms_medians`: a dict `{"unit-04": ..., "stanley": ...}` of their medians;
- `rms_ranges`: the same for their ranges (max − min);
- `stanley_better_beyond_noise`: `True` when Stanley's RMS median is lower than Unit 4's by more than both ranges;
- `times_noise_aware`: `noise_aware_better(stanley times, unit-04 times)` on the race times;
- `verdict`: one or two sentences: what can you honestly claim about Stanley against Unit 4 here, on the line and on the clock?

In [ ]:
unit_04_rms = []  # TODO
stanley_rms = []  # TODO
rms_medians = {}  # TODO
rms_ranges = {}  # TODO
stanley_better_beyond_noise = None  # TODO
times_noise_aware = None  # TODO
verdict = "TODO"

## Exercise 8 - Challenge: the best lookahead at lag 1

The lesson swept the lookahead at lag 0 (best `PP_K` = 0.2 s) and lag 2 (best 0.3 s).
Fill in lag 1.

Race the submission with `TARGET_SPEED = 30.0` and `PP_K` ∈ {0.15, 0.2, 0.25, 0.3, 0.35, 0.4} on `"test-circuit"` with `KinematicCar(jitter=0.05, lag_ticks=1)`, `seed=0` and `max_seconds=400.0`.

Define:

- `rms_by_k`: a dict from `PP_K` to the RMS cross-track error (m);
- `clean_by_k`: a dict from `PP_K` to `True` when that race finished with 0 collisions;
- `best_k_lag1`: the `PP_K` with the lowest RMS among the clean races;
- `between`: `True` when `best_k_lag1` lies between the lesson's lag-0 and lag-2 optima (inclusive);
- `one_seed_caveat`: one sentence on what you would have to race before trusting `best_k_lag1` as a tuning, by Unit 1's rule.

In [ ]:
rms_by_k = {}  # TODO
clean_by_k = {}  # TODO
best_k_lag1 = None  # TODO
between = None  # TODO
one_seed_caveat = "TODO"

## Exercise 9 - Challenge: a target exactly L_d along the track

The submission's target is the first **waypoint** at a **straight-line** distance of at least L_d.
Two things make it look further than L_d: the waypoints are about 2 m apart, so the target overshoots by up to a spacing, and in a bend the straight-line distance is shorter than the distance along the track.

Write `interpolated_target(xs, ys, segment, px, py, lookahead)`: project the car onto segment `segment` (the foot clamped to the segment), then walk forward along the polyline from that foot, segment by segment (cyclically), and return the point (x, y) at **arc length exactly L_d** from the foot, interpolating inside the segment where it lands.

On `test-circuit`, for every 10th waypoint i (i = 0, 10, 20, …), put a car 0.3 m to the left of waypoint i, heading along segment i, with `segment = i` and L_d = 9 m.
For each pose compute the pure-pursuit wheel angle toward the first-waypoint target (Exercise 2's `first_target`) and toward the interpolated target.

Define:

- `interpolated_target`: the function;
- `delta_first`, `delta_interpolated`: NumPy arrays of the two wheel angles (rad), one per pose;
- `max_delta_gap`: the largest |difference| between them (rad);
- `mean_extra_distance`: the mean, over the poses, of (straight-line distance to the first-waypoint target − L_d) (m);
- `worth_it`: one or two sentences: does the interpolated target change the steering enough to matter, and where on the track is the gap largest?

In [ ]:
def interpolated_target(xs, ys, segment, px, py, lookahead):
    return None  # TODO: (x, y)


delta_first = None  # TODO
delta_interpolated = None  # TODO
max_delta_gap = None  # TODO
mean_extra_distance = None  # TODO
worth_it = "TODO"